# Ciringoni M3 evaluasi checkpoint historis

Jalankan inferensi dan evaluasi E0, E5 P2, dan E6 mosaic nonaktif pada CPU Kaggle. Bobot berasal dari pelatihan lokal lima epoch yang tercatat pada 17 November 2026. Notebook ini memakai 356 gambar validation dan 2.205 anotasi.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
from zipfile import ZipFile

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
OUTPUT = WORK / "m3-reference"
OUTPUT.mkdir()
REPO = "https://github.com/Jordyee/ciringoni-data-mining.git"
SOURCE = "ee8c14bd132a07db3dae4bb15de0d8a22f517d60"
PROJECT = Path("/tmp/ciringoni-source/Data_Mining_Ciringoni_M1")
SCRIPTS = PROJECT / "hasil_m2"

def execute(*args):
    subprocess.run([str(arg) for arg in args], check=True, cwd=PROJECT)

def input_file(name):
    paths = list(INPUT.rglob(name))
    assert len(paths) == 1, f"Expected one mounted {name}, found {len(paths)}"
    return paths[0]

subprocess.run(["git", "clone", "--quiet", REPO, str(Path("/tmp/ciringoni-source"))], check=True)
execute("git", "checkout", "--quiet", SOURCE)
execute(sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.210", "pycocotools==2.0.11")
archives = list(INPUT.rglob("processed_train_val.zip"))
assert len(archives) <= 1
ROOT = Path("/tmp/ciringoni-processed/dataset")
if archives:
    with ZipFile(archives[0]) as archive:
        archive.extractall(ROOT.parent)
else:
    inputs = [path.parent.parent for path in INPUT.rglob("train")
              if path.is_dir() and path.parent.name == "images" and (path.parent.parent / "labels/val").is_dir()]
    assert len(inputs) == 1, f"Expected one processed input root, found {len(inputs)}"
    shutil.copytree(inputs[0], ROOT)
for split, expected in (("train", 1244), ("val", 356)):
    images = {path.stem for path in (ROOT / "images" / split).glob("*.jpg")}
    labels = {path.stem for path in (ROOT / "labels" / split).glob("*.txt")}
    assert len(images) == expected and images == labels
    print(f"{split}: {len(images)} JPG and labels")
assert not (ROOT / "images/test").exists()
YAML = ROOT / "data.yaml"
YAML.write_text(json.dumps({"path": str(ROOT), "train": "images/train", "val": "images/val", "names": ["infected"]}), encoding="utf-8")
for target, name in [(PROJECT / "data", "yolo26n.pt"), (PROJECT / "hasil_m1", "coco_val_gt.json")]:
    target.mkdir(exist_ok=True)
    shutil.copyfile(input_file(name), target / name)
with (PROJECT / "data/yolo26n.pt").open("rb") as stream:
    WEIGHTS_SHA = hashlib.file_digest(stream, "sha256").hexdigest()
assert WEIGHTS_SHA == "9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef"
from tarfile import open as tar_open
bundles = list(INPUT.rglob("reference-checkpoints.tar"))
if bundles:
    assert len(bundles) == 1
    with tar_open(bundles[0]) as packed:
        packed.extractall(Path("/tmp/ciringoni-reference"), filter="data")

def reference_manifest(name):
    files = list(INPUT.rglob(name)) + list(Path("/tmp/ciringoni-reference").rglob(name))
    assert len(files) == 1, f"Expected one {name} in reference artifacts"
    return files[0]


## Verifikasi checkpoint

Cocokkan SHA-256 tiap checkpoint beserta jumlah baris hasil latihan. Semua metrik berikut dihitung ulang di Kaggle.


In [ ]:
import os, platform
import torch, ultralytics
from pathlib import Path

RUNS = ("E0_640", "E5_640_p2", "E6_640_no_mosaic")
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
torch.set_num_threads(4)
reference = reference_manifest("m3-training-reference.json")
history = json.loads(reference.read_text(encoding="utf-8"))
EPOCHS = history["epochs"]
assert not history["trained_on_kaggle"] and EPOCHS == 5 and history["batch"] == 8 and history["seed"] == 42
assert set(history["runs"]) == set(RUNS)
TRAINING = history["runs"]
for run, data in TRAINING.items():
    checkpoint = reference.parent / data["path"]
    with checkpoint.open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == data["best_pt_sha256"]
    assert len((checkpoint.parents[1] / "results.csv").read_text().splitlines()) == EPOCHS + 1
    target = SCRIPTS / data["path"]
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(checkpoint, target)
    print(f"Historical checkpoint verified: {run} | {data['best_pt_sha256']}", flush=True)
shutil.copyfile(reference, OUTPUT / "m3-training-reference.json")
try:
    cpu_model = next(line for line in Path("/proc/cpuinfo").read_text().splitlines() if line.startswith("model name")).split(":", 1)[1].strip()
except (OSError, StopIteration):
    cpu_model = platform.processor()
print("Kaggle CPU runtime:", cpu_model, torch.__version__, ultralytics.__version__)


## Metrik validation dan latensi CPU

AP-small memakai COCOeval. mAP di CSV memakai validator Ultralytics. Benchmark CPU menetapkan empat thread Torch dan memanaskan 10 gambar.


In [ ]:
import pandas as pd
execute(sys.executable, SCRIPTS / "evaluasi_mosaic.py", YAML, "--device", "cpu", "--runs", *RUNS)
execute(sys.executable, SCRIPTS / "evaluasi_mosaic.py", YAML, "--device", "cpu", "--runs", *RUNS, "--cpu-benchmark")
for run in RUNS:
    shutil.copytree(SCRIPTS / "evaluasi" / run, OUTPUT / "evaluation" / run)
METRICS = OUTPUT / "eksperimen.csv"
shutil.copyfile(SCRIPTS / "evaluasi/eksperimen_mosaic_val.csv", METRICS)
rows = pd.read_csv(METRICS).set_index("id").loc[list(RUNS)]
assert rows["gambar"].eq(356).all()
display(rows[["AP-small", "mAP50", "mAP50-95", "presisi_conf_0.05_iou_0.5", "recall_conf_0.05_iou_0.5", "cpu_median_ms", "cpu_p95_ms"]])


## Analisis kesalahan dan seleksi ablation

Hitung kotak kecil yang terlewat pada confidence 0,05 dan IoU 0,50. Keputusan hanya berlaku untuk checkpoint historis olahan 640 yang dibandingkan di sini.


In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

GT = COCO(str(PROJECT / "hasil_m1/coco_val_gt.json"))
assert len(GT.imgs) == 356 and len(GT.anns) == 2205
errors = []
predictions_by_run = {}
by_image = {}
for run in RUNS:
    predictions = json.loads((OUTPUT / "evaluation" / run / "coco_val_predictions.json").read_text())
    retained = [item for item in predictions if item["score"] >= 0.05]
    predictions_by_run[run] = defaultdict(list)
    for item in retained:
        predictions_by_run[run][item["image_id"]].append(item)
    if retained:
        scorer = COCOeval(GT, GT.loadRes(retained), "bbox")
        scorer.params.iouThrs = np.array([0.50])
        scorer.params.areaRng = [scorer.params.areaRng[0]]
        scorer.params.areaRngLbl = ["all"]
        scorer.evaluate()
        matched_gt = {int(value) for item in scorer.evalImgs if item is not None
                      for value in item["dtMatches"][0][~item["dtIgnore"][0]] if value}
    else:
        matched_gt = set()
    images = defaultdict(lambda: [0, 0])
    for ann in GT.anns.values():
        subset = int(min(ann["bbox"][2:]) >= 8)
        images[ann["image_id"]][subset] += int(ann["id"] not in matched_gt)
    by_image[run] = images
    small = {ann["id"] for ann in GT.anns.values() if min(ann["bbox"][2:]) < 8}
    tp = len(matched_gt)
    errors.append({"run": run, "tp": tp, "fp": len(retained) - tp, "fn": len(GT.anns) - tp,
                   "gt_short_side_lt_8px": len(small), "tp_short_side_lt_8px": len(small & matched_gt)})
errors = pd.DataFrame(errors)
errors.to_csv(OUTPUT / "analisis_kesalahan.csv", index=False)
display(errors)

control = rows.loc["E0_640"]
mbr = "recall_conf_0.05_iou_0.5"
choices = []
for run in RUNS[1:]:
    row = rows.loc[run]
    change = {"AP-small": float(row["AP-small"] - control["AP-small"]),
              "mAP50-95": float(row["mAP50-95"] - control["mAP50-95"]),
              "recall": float(row[mbr] - control[mbr])}
    accepted = change["AP-small"] >= 0.005 and change["mAP50-95"] >= -0.005 and change["recall"] >= -0.02
    choices.append({"run": run, "delta_vs_E0_640": change, "eligible": accepted})
eligible = [choice["run"] for choice in choices if choice["eligible"]]
if eligible:
    top_ap = max(rows.loc[run, "AP-small"] for run in eligible)
    near = [run for run in eligible if top_ap - rows.loc[run, "AP-small"] < 0.005]
    selection = min(near, key=lambda run: rows.loc[run, "cpu_p95_ms"])
else:
    selection = "E0_640"
metadata = {"source_commit": SOURCE, "initial_weights_sha256": WEIGHTS_SHA, "train_images": 1244, "validation_images": 356, "validation_boxes": 2205,
            "epochs": EPOCHS, "batch": 8, "seed": 42, "device": cpu_model,
            "evaluation_performed_on_kaggle": True, "training_performed_on_kaggle": False,
            "checkpoint_origin": "Historical 2026-11-17 local training",
            "torch": str(torch.__version__), "ultralytics": ultralytics.__version__,
            "training": TRAINING, "matched_control": "E0_640", "choices": choices, "selected_m3_640": selection}
(OUTPUT / "manifest.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(json.dumps(metadata, indent=2))

image_id = max(GT.imgs, key=lambda index: (by_image["E0_640"][index][0], by_image["E0_640"][index][1], -index))
image = Image.open(ROOT / "images/val" / GT.imgs[image_id]["file_name"])
fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
for axis, title in zip(axes, ("Anotasi acuan", *RUNS), strict=True):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
    detections = GT.loadAnns(GT.getAnnIds(imgIds=[image_id])) if title == "Anotasi acuan" else predictions_by_run[title][image_id]
    for item in detections:
        x, y, w, h = item["bbox"]
        axis.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=1.1,
                                 edgecolor="lime" if title == "Anotasi acuan" else "orange"))
fig.savefig(OUTPUT / "contoh_kesalahan_val.png", dpi=140)
plt.show()
